# 01 — Load and clean
Builds the play and drive tables. All definitions are in `question-doc.md`.

## Load


In [1]:
from pathlib import Path
import pandas as pd
import nflreadpy as nfl

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)
RAW_PATH = DATA_DIR / "pbp_2016_2025_raw.parquet"

if RAW_PATH.exists():
    pbp = pd.read_parquet(RAW_PATH)
else:
    pbp = nfl.load_pbp(seasons=list(range(2016, 2026))).to_pandas()
    pbp.to_parquet(RAW_PATH)

pbp.shape


(484254, 372)

## Scope (§3)

In [2]:
n0 = len(pbp)
df = pbp[pbp["season_type"] == "REG"]
n1 = len(df)
df = df[df["game_half"].isin(["Half1", "Half2"])]
n2 = len(df)
df = df[(df["qb_kneel"] != 1) & (df["qb_spike"] != 1)]
n3 = len(df)
df = df[df["two_point_attempt"] != 1].copy()

pd.Series({"raw": n0, "regular season": n1, "no overtime": n2,
           "no kneels/spikes": n3, "no 2-pt tries": len(df)})

raw                 484254
regular season      462773
no overtime         459667
no kneels/spikes    454898
no 2-pt tries       453673
dtype: int64

## Time buckets (§4)

In [3]:
# intervals are (low, high], so a play exactly on a boundary lands in the later bucket
bins   = [0, 60, 120, 180, 240, 480, 900, 1800]
labels = ["B6", "B5", "B4", "B3", "B2", "B1", "B0"]

df["bucket"] = pd.cut(df["half_seconds_remaining"], bins=bins, labels=labels, include_lowest=True)
df["is_late"] = df["bucket"].isin(["B3", "B4", "B5", "B6"])

print(df["bucket"].isna().sum(), "rows with no bucket")
df["bucket"].value_counts().reindex(["B0", "B1", "B2", "B3", "B4", "B5", "B6"])

208 rows with no bucket


bucket
B0    198425
B1    100101
B2     55035
B3     16383
B4     17370
B5     29179
B6     36972
Name: count, dtype: int64

## Play table (§3, §5)
Runs and passes with an EPA value. Non-plays, punts, kicks etc. drop out here, but stay in `df` for the drive table.

In [4]:
plays = df[df["play_type"].isin(["pass", "run"]) & df["epa"].notna()].copy()
plays["success"] = plays["epa"] > 0

len(plays), plays["success"].mean()

(326039, np.float64(0.4360153233202163))

In [5]:
nb = df[df["bucket"].isna()]
print(nb["half_seconds_remaining"].describe())
nb["play_type"].value_counts(dropna=False)

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: half_seconds_remaining, dtype: float64


play_type
NaN        206
no_play      2
Name: count, dtype: int64

## Drive table (§4, §5)
First, inspect how nflverse labels drives.

In [6]:
[c for c in df.columns if "drive" in c]

['drive',
 'fixed_drive',
 'fixed_drive_result',
 'drive_real_start_time',
 'drive_play_count',
 'drive_time_of_possession',
 'drive_first_downs',
 'drive_inside20',
 'drive_ended_with_score',
 'drive_quarter_start',
 'drive_quarter_end',
 'drive_yards_penalized',
 'drive_start_transition',
 'drive_end_transition',
 'drive_game_clock_start',
 'drive_game_clock_end',
 'drive_start_yard_line',
 'drive_end_yard_line',
 'drive_play_id_started',
 'drive_play_id_ended']

In [7]:
g = df[df["game_id"] == df["game_id"].iloc[0]]
cols = ["play_id", "qtr", "half_seconds_remaining", "posteam", "drive",
        "fixed_drive", "fixed_drive_result", "play_type", "down", "ydstogo"]
g[cols].head(40)

,play_id,qtr,half_seconds_remaining,posteam,drive,fixed_drive,fixed_drive_result,play_type,down,ydstogo
0,1.0,1.0,1800.0,NaN,NaN,1.0,Punt,NaN,NaN,0.0
1,36.0,1.0,1800.0,BAL,1.0,1.0,Punt,kickoff,NaN,0.0
2,58.0,1.0,1797.0,BAL,1.0,1.0,Punt,run,1.0,10.0
3,85.0,1.0,1772.0,BAL,1.0,1.0,Punt,pass,2.0,4.0
4,109.0,1.0,1741.0,BAL,1.0,1.0,Punt,run,1.0,10.0
5,130.0,1.0,1715.0,BAL,1.0,1.0,Punt,pass,2.0,10.0
6,154.0,1.0,1674.0,BAL,1.0,1.0,Punt,run,3.0,1.0
7,175.0,1.0,1625.0,BAL,1.0,1.0,Punt,no_play,4.0,1.0
8,198.0,1.0,1612.0,BAL,1.0,1.0,Punt,punt,4.0,6.0
9,216.0,1.0,1604.0,BUF,2.0,2.0,Punt,pass,1.0,10.0


## Drive table (§4, §5)
One row per drive. Start time and field position come from the first scrimmage play, not the kickoff row.

In [8]:
d = df[df["posteam"].notna() & df["fixed_drive"].notna()].sort_values(["game_id", "play_id"])

# first non-kickoff row of each drive = where the offense actually starts
scrim = d[d["play_type"].isin(["pass", "run", "no_play", "punt", "field_goal"])]

drives = scrim.drop_duplicates(["game_id", "fixed_drive"])[
    ["game_id", "season", "fixed_drive", "posteam", "game_half",
     "half_seconds_remaining", "bucket", "yardline_100", "fixed_drive_result"]
].rename(columns={
    "half_seconds_remaining": "start_secs",
    "bucket": "start_bucket",
    "yardline_100": "start_yardline",
    "fixed_drive_result": "result",
})

drives["is_td"] = drives["result"] == "Touchdown"
drives["eligible"] = drives["start_secs"] >= 30   # §4 drive start rule

print(len(drives), "drives")
drives["result"].value_counts(dropna=False)

55917 drives


result
Punt                 21318
Touchdown            12703
Field goal            8591
Turnover              5953
Turnover on downs     2862
End of half           2092
Missed field goal     1549
Opp touchdown          695
Safety                 150
NaN                      4
Name: count, dtype: int64

### Drop unreliable drives
Drives where nflverse mixes two teams' plays (19), and drives with no result (4).

In [9]:
multi = scrim.groupby(["game_id", "fixed_drive"])["posteam"].nunique()
multi = multi[multi > 1].index

mixed = pd.MultiIndex.from_frame(drives[["game_id", "fixed_drive"]]).isin(multi)
drives = drives[~mixed].dropna(subset=["result"]).reset_index(drop=True)

print(len(drives), "drives")

55894 drives


In [10]:
print("drives per game:", len(drives) / drives["game_id"].nunique())
drives[drives["eligible"]].groupby("game_half")["is_td"].mean()

drives per game: 21.179992421371733


game_half
Half1    0.230370
Half2    0.231735
Name: is_td, dtype: float64

## Save

In [11]:
plays.to_parquet(DATA_DIR / "plays.parquet")
drives.to_parquet(DATA_DIR / "drives.parquet")